# DonkeyCar Experiment Analysis

Analysis of autonomous DonkeyCar autopilot tuning results from `results_donkey.tsv`.

**Reference baseline**: wroscoe's run (14 Mar 2026) achieved CTE ≈ 2.74 on an unseen track
after overnight iteration. Our metric is `val_cte` (mean absolute cross-track error, lower is better).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated: commit, val_cte, memory_gb, status, description)
df = pd.read_csv("results_donkey.tsv", sep="\t")
df["val_cte"] = pd.to_numeric(df["val_cte"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    cte = row["val_cte"]
    desc = row["description"]
    print(f"  #{i:3d}  cte={cte:.4f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Val CTE Over Time

Track how the best (kept) `val_cte` evolves as experiments progress.
The running minimum shows the frontier — the best result achieved so far.
The dashed line marks wroscoe's reference CTE of 2.74 from his overnight run.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# wroscoe's reference CTE (snapshot from his eval on an unseen track)
WROSCOE_CTE = 2.74

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

if len(valid) == 0:
    print("No valid experiments to plot.")
else:
    baseline_cte = valid.loc[0, "val_cte"]
    best_cte = valid["val_cte"].min()

    # Plot discarded as faint background dots
    disc = valid[valid["status"] == "DISCARD"]
    ax.scatter(disc.index, disc["val_cte"],
               c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

    # Plot kept experiments as prominent blue dots
    kept_v = valid[valid["status"] == "KEEP"]
    ax.scatter(kept_v.index, kept_v["val_cte"],
               c="#3498db", s=50, zorder=4, label="Kept",
               edgecolors="black", linewidths=0.5)

    # Running minimum step line
    kept_mask = valid["status"] == "KEEP"
    kept_idx = valid.index[kept_mask]
    kept_cte = valid.loc[kept_mask, "val_cte"]
    if len(kept_cte) > 0:
        running_min = kept_cte.cummin()
        ax.step(kept_idx, running_min, where="post", color="#2980b9",
                linewidth=2, alpha=0.7, zorder=3, label="Running best")

    # wroscoe reference line
    ax.axhline(y=WROSCOE_CTE, color="#e74c3c", linestyle="--",
               linewidth=1.5, alpha=0.7, zorder=1,
               label=f"wroscoe ref CTE = {WROSCOE_CTE}")

    # Label each kept experiment with its description
    for idx, cte in zip(kept_idx, kept_cte):
        desc = str(valid.loc[idx, "description"]).strip()
        if len(desc) > 45:
            desc = desc[:42] + "..."
        ax.annotate(desc, (idx, cte),
                    textcoords="offset points",
                    xytext=(6, 6), fontsize=8.0,
                    color="#1a5276", alpha=0.9,
                    rotation=30, ha="left", va="bottom")

    n_total = len(df)
    n_kept = len(df[df["status"] == "KEEP"])
    ax.set_xlabel("Experiment #", fontsize=12)
    ax.set_ylabel("Validation CTE (lower is better)", fontsize=12)
    ax.set_title(f"DonkeyCar AutoResearch Progress: {n_total} Experiments, "
                 f"{n_kept} Kept Improvements", fontsize=14)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.2)

    # Y-axis: show the interesting region
    y_lo = min(best_cte, WROSCOE_CTE) * 0.9
    y_hi = baseline_cte * 1.1
    ax.set_ylim(y_lo, y_hi)

    plt.tight_layout()
    plt.savefig("progress_donkey.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to progress_donkey.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
if len(kept) == 0:
    print("No kept experiments yet.")
else:
    baseline_cte = df.iloc[0]["val_cte"]
    best_cte = kept["val_cte"].min()
    best_row = kept.loc[kept["val_cte"].idxmin()]

    print(f"Baseline val_cte:  {baseline_cte:.4f}")
    print(f"Best val_cte:      {best_cte:.4f}")
    print(f"Total improvement: {baseline_cte - best_cte:.4f} "
          f"({(baseline_cte - best_cte) / baseline_cte * 100:.2f}%)")
    print(f"Best experiment:   {best_row['description']}")
    print()

    # Compare against wroscoe
    WROSCOE_CTE = 2.74
    if best_cte <= WROSCOE_CTE:
        print(f"✅ Our best CTE ({best_cte:.4f}) beats wroscoe's reference ({WROSCOE_CTE})")
    else:
        gap = best_cte - WROSCOE_CTE
        print(f"⬇️  Our best CTE ({best_cte:.4f}) is {gap:.4f} above wroscoe's reference ({WROSCOE_CTE})")
    print()

    # Cumulative effort per improvement
    print("Cumulative effort per improvement:")
    kept_sorted = kept.reset_index()
    for i, (_, row) in enumerate(kept_sorted.iterrows()):
        desc = str(row["description"]).strip()
        print(f"  Experiment #{row['index']:3d}: cte={row['val_cte']:.4f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
if len(kept) < 2:
    print("Need at least 2 kept experiments to rank improvements.")
else:
    kept["prev_cte"] = kept["val_cte"].shift(1)
    kept["delta"] = kept["prev_cte"] - kept["val_cte"]

    # Drop baseline (no delta)
    hits = kept.iloc[1:].copy()
    hits = hits.sort_values("delta", ascending=False)

    print(f"{'Rank':>4}  {'Delta':>8}  {'CTE':>10}  Description")
    print("-" * 80)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.4f}  {row['val_cte']:.4f}      {row['description']}")

    print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>10}  "
          f"TOTAL improvement over baseline")